# Smartwatch Health Analytics — Improved Stress Prediction

This notebook uses the cleaned ML-ready smartwatch dataset.

## Goals
1. Avoid aggressive IQR row deletion.
2. Clean categorical inconsistencies.
3. Engineer health/stress-related features.
4. Compare a 10-class stress model with a 3-class business model.
5. Evaluate Accuracy, Precision, Recall and F1-score.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.ensemble import RandomForestClassifier

# XGBoost
# Install if required:
# !pip install xgboost
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings("ignore")


In [ ]:
# Load the improved dataset
df = pd.read_csv("Smartwatch_Health_Analytics_ML_Ready.csv")

print("Shape:", df.shape)
display(df.head())
display(df.info())


In [ ]:
# Target distribution
print(df["Stress Level"].value_counts().sort_index())

df["Stress Level"].value_counts().sort_index().plot(kind="bar")
plt.title("Stress Level Distribution")
plt.xlabel("Stress Level")
plt.ylabel("Number of Records")
plt.show()


In [ ]:
# Separate target and features
target = "Stress Level"

X = df.drop(columns=[target, "Stress Band"], errors="ignore")
y = df[target].astype(int)

print("Features:", X.shape)
print("Target:", y.shape)


In [ ]:
# Identify numeric and categorical features
numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", categorical_features)


In [ ]:
# Preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


In [ ]:
# Baseline Random Forest
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print("Random Forest")
print("Accuracy :", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred, average="macro", zero_division=0))
print("Recall   :", recall_score(y_test, rf_pred, average="macro", zero_division=0))
print("F1-score :", f1_score(y_test, rf_pred, average="macro", zero_division=0))

print("\nClassification Report:")
print(classification_report(y_test, rf_pred, zero_division=0))


In [ ]:
# XGBoost — 10-class stress prediction
xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="multi:softmax",
        num_class=10,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    ))
])

xgb_model.fit(X_train, y_train - 1)  # XGBoost multiclass labels: 0–9

xgb_pred = xgb_model.predict(X_test).astype(int) + 1

print("XGBoost — 10 Stress Classes")
print("Accuracy :", accuracy_score(y_test, xgb_pred))
print("Precision:", precision_score(y_test, xgb_pred, average="macro", zero_division=0))
print("Recall   :", recall_score(y_test, xgb_pred, average="macro", zero_division=0))
print("F1-score :", f1_score(y_test, xgb_pred, average="macro", zero_division=0))

print("\nClassification Report:")
print(classification_report(y_test, xgb_pred, zero_division=0))


In [ ]:
# Confusion matrix — 10 classes
cm = confusion_matrix(y_test, xgb_pred, labels=list(range(1, 11)))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=list(range(1, 11))
)
disp.plot()
plt.title("XGBoost — Stress Level 1–10")
plt.show()


In [ ]:
# Business-oriented target: Low / Moderate / High
df["Stress Band"] = pd.cut(
    df["Stress Level"],
    bins=[0, 3, 7, 10],
    labels=["Low", "Moderate", "High"],
    include_lowest=True
)

X3 = df.drop(columns=["Stress Level", "Stress Band"], errors="ignore")
y3 = df["Stress Band"].astype(str)

num3 = X3.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
cat3 = X3.select_dtypes(include=["object", "string", "category"]).columns.tolist()

prep3 = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num3),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), cat3)
])

X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3,
    test_size=0.20,
    random_state=42,
    stratify=y3
)

rf3 = Pipeline([
    ("preprocessor", prep3),
    ("model", RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

rf3.fit(X3_train, y3_train)
pred3 = rf3.predict(X3_test)

print("Business Model — Low / Moderate / High")
print("Accuracy :", accuracy_score(y3_test, pred3))
print("Precision:", precision_score(y3_test, pred3, average="macro", zero_division=0))
print("Recall   :", recall_score(y3_test, pred3, average="macro", zero_division=0))
print("F1-score :", f1_score(y3_test, pred3, average="macro", zero_division=0))

print("\nClassification Report:")
print(classification_report(y3_test, pred3, zero_division=0))


In [ ]:
# Confusion matrix — business model
cm3 = confusion_matrix(
    y3_test,
    pred3,
    labels=["Low", "Moderate", "High"]
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm3,
    display_labels=["Low", "Moderate", "High"]
)
disp.plot()
plt.title("Stress Band Prediction")
plt.show()


In [ ]:
# Compare models
results = pd.DataFrame({
    "Model": [
        "Original model (from project)",
        "Improved Random Forest — 10 classes",
        "Improved XGBoost — 10 classes",
        "Business Random Forest — 3 classes"
    ],
    "Target": [
        "Stress 1–10",
        "Stress 1–10",
        "Stress 1–10",
        "Low / Moderate / High"
    ],
    "Accuracy": [
        0.1845,
        accuracy_score(y_test, rf_pred),
        accuracy_score(y_test, xgb_pred),
        accuracy_score(y3_test, pred3)
    ],
    "Macro Precision": [
        np.nan,
        precision_score(y_test, rf_pred, average="macro", zero_division=0),
        precision_score(y_test, xgb_pred, average="macro", zero_division=0),
        precision_score(y3_test, pred3, average="macro", zero_division=0)
    ],
    "Macro Recall": [
        np.nan,
        recall_score(y_test, rf_pred, average="macro", zero_division=0),
        recall_score(y_test, xgb_pred, average="macro", zero_division=0),
        recall_score(y3_test, pred3, average="macro", zero_division=0)
    ],
    "Macro F1": [
        np.nan,
        f1_score(y_test, rf_pred, average="macro", zero_division=0),
        f1_score(y_test, xgb_pred, average="macro", zero_division=0),
        f1_score(y3_test, pred3, average="macro", zero_division=0)
    ]
})

display(results.round(4))


## Key conclusions

- Do **not** use sequential IQR filtering on every numerical column.
- Do not numerically label unrelated categorical values.
- Use OneHotEncoder for categorical features.
- Keep the original 1–10 target for detailed prediction.
- Use Low / Moderate / High as a separate business target when the goal is actionable stress monitoring.
- Do not artificially modify target labels just to increase accuracy.
- Compare models using macro precision, recall and F1—not accuracy alone.
